In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from pathlib import Path
from copy import deepcopy

from openquake.hazardlib.imt import PGA, SA, RSD595, AvgSA, IMT

from pickagm.distributions import ensemble_ks_bounds

from phd_project.config.config import load_config 
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    ESHM20SiteRupCtxBuilder,
    create_gmm_map,
    create_corr_model_map,
    calculate_gcim_distributions_for_sites,
    get_ground_motion_ensembles_for_sites,
    get_best_ensemble_in_list,
    total_ks_statistic_and_failing_im_penalty,
    optimise_ground_motion_ensembles_for_sites,
    optimise_ground_motion_ensembles_for_sites_with_shuffles,
)
import phd_project.scripts.WP1_ground_motion_set.manage_flatfiles as mf
from phd_project.plotting import custom_log_formatter

cfg = load_config()

C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\gm_selection.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# AvgSA([0, 3])

In [3]:
# Load the disaggregation data
fp = cfg["proc_data"]["site_hazard"] / "AvgSA_03_disagg_data_60sites.pickle"
with open(fp, "rb") as f:
    disagg_data = pickle.load(f)

fp = cfg["proc_data"]["site_hazard"] / "AvgSA_03_disagg_stats_60sites.pickle"
with open(fp, "rb") as f:
    disagg_stats = pickle.load(f)

# load the site file
sites = pd.read_csv(cfg["hazard_models"]["eshm20_AvgSA_site_model_all"])

# load the flatfiles
flatfile_folder = cfg["proc_data"]["corr_model"] / "reverse" / "flatfiles"
flatfiles = {}
for f in [f for f in os.listdir(flatfile_folder) if f.endswith(".csv")]:
    tag = f.split("_")[0]
    flatfiles[tag] = pd.read_csv(flatfile_folder / f, delimiter=";", index_col=0, low_memory=False)

flatfiles["volcanic"] = pd.read_csv(cfg["raw_data"]["gm_flatfiles"] / "volcanic_lanzanoluzi_flatfile.csv", 
                                    delimiter=";", index_col=0)

# load the preprocessed gm database
gm_database = pd.read_csv(cfg["proc_data"]["gm_database"], sep=",", low_memory=False, header=[0, 1])

In [4]:
# create the average depth map for each TRT #TODO:: if this needs to be more specific
average_depths = {
    "Craton": flatfiles["asc"]["ev_depth_km"].mean(),
    "Non-Subduction Deep": flatfiles["vran"]["ev_depth_km"].mean(),
    "Shallow Default": flatfiles["asc"]["ev_depth_km"].mean(),
    "Subduction Inslab": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Subduction Interface": flatfiles["sinter"]["ev_depth_km"].mean(),
    "Volcanic": flatfiles["volcanic"]["ev_depth_km"].mean(),
}

# the map of what sim trts are allowed to match with what record trts
OK_TRT_MATCHES = {
    "Craton": ["Shallow Default"],
    "Non-Subduction Deep": ["Non-Subduction Deep", "Subduction Inslab", "Subduction"],
    "Shallow Default": ["Shallow Default"],
    "Subduction Inslab": ["Non-Subduction Deep", "Subduction Inslab", "Subduction"],
    "Subduction Interface": ["Subduction Interface"],
    "Volcanic": ["Shallow Default"],
}

occurence = True    # the record selection should be performed based on occurence

In [5]:
# set some parameters for the selection
t_lower = 0.025     # lower SA period considered in selection
t_upper = 3         # upper SA period considered in selection
n_periods = 20      # number of periods to consider in selection

conditioning_imt: IMT = AvgSA([0,3])                      
nonSA_imts: list[IMT] = [AvgSA([0,3]), RSD595(), PGA()] 
sa_periods = np.round(np.geomspace(t_lower, t_upper, num=n_periods), 3)
SA_imts: list[IMT] = [SA(period) for period in sa_periods]
selection_imts: list[IMT] = nonSA_imts[1:] + SA_imts
nonSA_imt_strs: list[str] = [im.string for im in nonSA_imts] # strings match the correlation matrix

# weights of the IMs
weight_rsd595 = 0.25
n_other_ims = len([imt for imt in selection_imts if imt.name == "SA" or imt.name == "PGA"])
imt_weights = np.array([(1-weight_rsd595) / n_other_ims if imt.name != "RSD595" 
                        else weight_rsd595 for imt in selection_imts])
imt_weights /= imt_weights.sum()

# some other things
disagg_type = "TRT_Mag_Dist_Eps"
percentiles = [0.05, 0.16, 0.5, 0.84, 0.95]     # percentiles of the gcim distribution to return  
assumed_rake = -90                              # assumed rake for RSD595 calculation

In [6]:
# filter the gm_database so that only the selection and conditioning ims are present
gm_db = gm_database.copy()
updated_ims = mf.filter_gm_database_on_imts(
    gm_db["ims"], selection_imts + [conditioning_imt])
updated_ims.columns = pd.MultiIndex.from_product([['ims'], updated_ims.columns])
gm_db = pd.concat([gm_db.drop('ims', axis=1, level=0), updated_ims], axis=1)

with open(r"C:\Users\clemettn\Documents\gm_db.pickle", "wb") as file:
    pickle.dump(gm_db, file)
    
# Create the GMM Map for AvgSA by reading the logic tree
AvgSA_03_lt_fp = cfg["hazard_models"]["eshm20_AvgSA"] / "gmpe_logic_tree_AvgSA_0to3_median_branch.xml"
gmm_map = create_gmm_map(AvgSA_03_lt_fp)

# get the correlation model map
corr_map = create_corr_model_map(nonSA_imt_strs, sa_periods)

# organise the disagg data
site_poe_disaggs = {}
for (s, r) in disagg_data.keys():
    for site in disagg_data[(s,r)].keys():
        for poe in disagg_data[(s,r)][site][conditioning_imt.name].keys():
            site_poe_disaggs[(site, poe)] = disagg_data[(s,r)][site][conditioning_imt.name][poe]
site_poes = sorted(list(site_poe_disaggs.keys()), key=lambda x: x[0])

In [10]:
# set up the selection context:
basic_selection_ctx = {
    "n_ensembles": 20,
    "n_samples": 30,
    "conditioning_imt": conditioning_imt ,
    "disagg_imt": conditioning_imt.name , # this only works for AvgSA. otherwise used .string 
    "selection_imts": selection_imts ,
    "imt_weights": imt_weights ,
    "sites": sites ,
    "ctx_builder": ESHM20SiteRupCtxBuilder ,
    "ctx_builder_params": ["average_depths", "assumed_rake"] ,
    "average_depths": average_depths ,
    "assumed_rake": assumed_rake ,
    "gmm_map": gmm_map ,
    "corr_map": corr_map ,
    "m_bound_model": "tarbali_and_bradley_2016" ,
    "d_bound_model": "tarbali_and_bradley_2016" ,
    "vs30_bound_model": "tarbali_and_bradley_2016" ,
    "sf_bounds": (0.25, 4),
    "usable_T": t_upper ,
    "max_n_recs": 5 ,
    "p_value": 0.05 ,
    "ok_trt_matches": OK_TRT_MATCHES ,
    "occurence": True ,
}

rng_seed = 1
LOAD_GCIM_IF_EXISTS = True
LOAD_RS_IF_EXISTS = True
LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS = True

## GCIM Target Distributions

In [11]:
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / f"gcim_dist_AvgSA_03_rake{int(assumed_rake)}.pickle"

if LOAD_GCIM_IF_EXISTS: # load the gcim distributions instead of 
    no_file = False
    if gcim_dist_fp.is_file():
        with open(gcim_dist_fp, "rb") as file:
            gcim_dists = pickle.load(file)
        print("Existing GCIM distribution data loaded...")
    else:
        no_file = True
        print("No existing GCIM distribution data found...")

if (not LOAD_GCIM_IF_EXISTS) or no_file:
    # calculate the gcims and save them
    gcim_dists = calculate_gcim_distributions_for_sites(
        site_poe_disaggs, disagg_stats, conditioning_imt, 
        selection_imts, sites, gmm_map, corr_map, 
        average_depths, assumed_rake, occurence, percentiles)

    with open(gcim_dist_fp, "wb") as file:
        pickle.dump(gcim_dists, file)

Existing GCIM distribution data loaded...


## Preliminary Selection

In [12]:
only_select = []

preliminary_selection_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_prelim_selection_rake{int(assumed_rake)}.pickle"

if LOAD_RS_IF_EXISTS: # load the preliminary record selection instead of calculating 
    no_file = False
    if preliminary_selection_fp.is_file():
        with open(preliminary_selection_fp, "rb") as file:
            preliminary_ensembles = pickle.load(file)
        print("Existing record selection data loaded...")
    else:
        no_file = True
        print("No existing record selection data found...")
        
if (not LOAD_RS_IF_EXISTS) or no_file:
    # do the record selection for all sites and save the results
    candidate_ensembles = get_ground_motion_ensembles_for_sites(
        site_poe_disaggs, disagg_stats, gcim_dists, 
        gm_db, basic_selection_ctx, sites, rng_seed, only_select)
    
    # get the best of the candidate ensembles and save
    obj_func = total_ks_statistic_and_failing_im_penalty

    preliminary_ensembles = {}

    
    for (site, poe), ensembles in candidate_ensembles.items():

        if ensembles == []:
            # no ensemble was found
            preliminary_ensembles[(site, poe)] = None
            continue

        target_cdfs = gcim_dists[(site, poe)]["cdfs"]
        ks_bounds = ensemble_ks_bounds(
            target_cdfs, 
            basic_selection_ctx["n_samples"],
            basic_selection_ctx["p_value"])
        
        obj_func_kwargs = {
            "target_cdfs_list": [np.column_stack([np.log(ksb[1:,0]), ksb[1:,2]]) 
                                for ksb in ks_bounds.values()],
            "upper_ks_bounds_list": [np.column_stack([np.log(ksb[1:,0]), ksb[1:,3]]) 
                                    for ksb in ks_bounds.values()],
            "lower_ks_bounds_list": [np.column_stack([np.log(ksb[1:,0]), ksb[1:,1]]) 
                                    for ksb in ks_bounds.values()],
            "n_recs": basic_selection_ctx["n_samples"],
            "penalty_constant": 10
        }

        e = get_best_ensemble_in_list(
            ensembles, basic_selection_ctx["conditioning_imt"].string, 
            obj_func, obj_func_kwargs)
        preliminary_ensembles[(site, poe)] = e

    # Save the results
    with open(preliminary_selection_fp, "wb") as file:
        pickle.dump(preliminary_ensembles, file)

# Check if all the sites and poes found a set of suitable ground motions
prelim_ensembles_not_passing = []
no_preliminary_ensembles = []
for (site, poe), ensemble in preliminary_ensembles.items():
    
    if ensemble is None:
        prelim_ensembles_not_passing.append((site, poe))
        no_preliminary_ensembles.append((site, poe))
    
    elif not ensemble["ks_passed"]:
        prelim_ensembles_not_passing.append((site, poe))

if len(prelim_ensembles_not_passing) == 0:
    print(f"OK! - Preliminary ensembles pass for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - Preliminary ensembles do not pass for {len(prelim_ensembles_not_passing)} combinations of site and poe")

if no_preliminary_ensembles:
    print(f"No preliminary ensembles found for {len(no_preliminary_ensembles)} combinations of site and poe")

No existing record selection data found...


Selecting GM Ensembles:   0%|          | 0/360 [00:00<?, ?it/s]

Sub-Optimal! - Preliminary ensembles do not pass for 64 combinations of site and poe
No preliminary ensembles found for 24 combinations of site and poe


In [13]:
no_preliminary_ensembles

[(30, np.float64(0.000201)),
 (30, np.float64(0.0001)),
 (46, np.float64(0.0001)),
 (52, np.float64(0.001)),
 (52, np.float64(0.000404)),
 (52, np.float64(0.000201)),
 (52, np.float64(0.0001)),
 (55, np.float64(0.000404)),
 (55, np.float64(0.000201)),
 (55, np.float64(0.0001)),
 (56, np.float64(0.000404)),
 (56, np.float64(0.000201)),
 (56, np.float64(0.0001)),
 (57, np.float64(0.000404)),
 (57, np.float64(0.000201)),
 (57, np.float64(0.0001)),
 (58, np.float64(0.000404)),
 (58, np.float64(0.000201)),
 (58, np.float64(0.0001)),
 (59, np.float64(0.001)),
 (59, np.float64(0.000404)),
 (59, np.float64(0.000201)),
 (59, np.float64(0.0001)),
 (22, np.float64(0.0001))]

In [14]:
for (site, poe) in sorted(prelim_ensembles_not_passing):
    e = preliminary_ensembles[(site, poe)]
    if e == None:
        continue
    failing_ims = e["ks_failed_ims"]
    print(site, poe, failing_ims)

22 0.000201 ['RSD595', 'SA(0.4)', 'SA(0.514)', 'SA(0.662)', 'SA(0.851)', 'SA(1.095)', 'SA(2.332)']
22 0.000404 ['RSD595', 'SA(0.514)', 'SA(0.662)']
22 0.001 ['RSD595']
22 0.002103 ['RSD595']
22 0.004988 ['RSD595']
23 0.0001 ['SA(0.041)', 'SA(3.0)']
27 0.000201 ['RSD595']
27 0.001 ['RSD595']
27 0.002103 ['RSD595']
27 0.004988 ['RSD595']
28 0.0001 ['RSD595']
28 0.000201 ['RSD595']
28 0.000404 ['RSD595']
28 0.001 ['RSD595']
28 0.002103 ['RSD595']
28 0.004988 ['RSD595']
35 0.0001 ['RSD595']
36 0.0001 ['RSD595', 'PGA', 'SA(0.025)', 'SA(0.4)']
37 0.0001 ['RSD595', 'PGA', 'SA(0.4)']
40 0.0001 ['PGA', 'SA(0.4)']
41 0.0001 ['RSD595']
45 0.0001 ['PGA', 'SA(0.4)']
49 0.0001 ['RSD595']
51 0.0001 ['SA(0.4)']
52 0.002103 ['RSD595', 'PGA']
52 0.004988 ['RSD595']
55 0.001 ['RSD595']
55 0.002103 ['RSD595', 'SA(1.095)']
55 0.004988 ['RSD595']
56 0.001 ['RSD595']
56 0.002103 ['RSD595']
56 0.004988 ['RSD595']
57 0.001 ['RSD595', 'SA(0.514)', 'SA(0.662)']
57 0.002103 ['RSD595']
57 0.004988 ['RSD595']
58 0.

## Optimisation of Ensembles

### Round 1

Optimisation using the same parameters as the original selection

In [ ]:
only_optimise = []

optimised_selection_rd1_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_optimised_selection_rd01_rake{int(assumed_rake)}.pickle"

if LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS: 
    no_file = False
    if optimised_selection_rd1_fp.is_file():
        with open(optimised_selection_rd1_fp, "rb") as file:
            optimised_ensembles = pickle.load(file)
        print("Existing optimised ensembles loaded...")
    else:
        no_file = True
        print("No existing optimised ensembles found...")

if (not LOAD_OPTIMISED_ENSEMBLES_IF_EXISTS) or no_file:
    # do the ensemble optimisation and save the results
    optimised_ensembles, _ = optimise_ground_motion_ensembles_for_sites(
        preliminary_ensembles, site_poe_disaggs, disagg_stats,
        gcim_dists, gm_db, basic_selection_ctx, sites, only_optimise)

    # Save the results
    with open(optimised_selection_rd1_fp, "wb") as file:
        pickle.dump(optimised_ensembles, file)

# Check if all the sites and poes found a set of suitable ground motions
optim_ensembles_not_passing_rd1 = []
for (site, poe), ensemble in optimised_ensembles.items():

    if ensemble is None:
        optim_ensembles_not_passing_rd1.append((site, poe))

    elif not ensemble["ks_passed"]:
        optim_ensembles_not_passing_rd1.append((site, poe))

if len(optim_ensembles_not_passing_rd1) == 0:
    print(f"OK! - Optimised ensembles pass for all combinations of site and poe")
else:
    print(f"Sub-Optimal! - Optimised ensembles do not pass for {len(optim_ensembles_not_passing_rd1)} combinations of site and poe")

# Post-Processing

In [ ]:
# Save the best ensembles in the results folder
RESULT_FOLDER = cfg["results"]["AvgSA_03_record_selection"]

final_ensembles = {}
ensemble_list = []
ensemble_id_tuples = []
ensemble_tags = []

for (site, poe), ensemble in optimised_ensembles.items():

    if not ensemble["ks_passed"]:
        print(f"No valid ensemble: Site {site} - poe = {poe}")
        continue
    
    id_tuple = (f"site_{site}", f"{poe:g}".replace(".", "pt"))
    tag = "__".join([str(i) for i in id_tuple])
    
    # Add the gcim_quantiles from the result
    ensemble["gcim_quantiles"] = gcim_dists[(site, poe)]["stats"]

    # calculate the trt %'s
    trt_stats = pd.DataFrame(
        ensemble["recs"].groupby(("metadata", "trt")).size())
    trt_stats.columns = ["count"]
    trt_stats["proportions"] = np.round(trt_stats["count"] / \
                                        trt_stats["count"].sum(), 3)
    ensemble["trt_stats"] = trt_stats

    ensemble_list.append(ensemble)
    ensemble_id_tuples.append(id_tuple)
    ensemble_tags.append(tag)
    final_ensembles[(site, poe)] = ensemble

    with open(RESULT_FOLDER / f"{tag}__gm_selection.pickle", "wb") as file:
        pickle.dump(ensemble, file)
            

## Identify the number of unique records and unique events across all sites

In [ ]:
# combine all ensembles into one dataframe
rec_dfs = []
for e in final_ensembles.values():
    rec_dfs.append(e["recs"])

all_recs = pd.concat(rec_dfs, axis=0)
events = all_recs.groupby(("metadata", "event_id")).size().sort_values(ascending=False)
events = pd.DataFrame(events, columns=["count"])
events["cum_sum"] = events["count"].cumsum() 
events["cum_%"] = np.round(events["cum_sum"] / events["count"].sum(), 4)

recs_and_events = all_recs.groupby([("metadata", "event_id"), ("metadata", "station_code"), ("metadata", "component")])\
    .size().sort_values(ascending=False)
recs_and_events = pd.DataFrame(recs_and_events, columns=["count"])
recs_and_events["cum_sum"] = recs_and_events["count"].cumsum() 
recs_and_events["cum_%"] = np.round(recs_and_events["cum_sum"] / recs_and_events["count"].sum(), 4)
# todo:: bar charts to show how the number of times each record and event were
# todo:: selected
recs_and_events

print(f"Number of unique events: {len(events)}")
print(f"Number of unique records (components counted separately): {len(recs_and_events)}")
print(f"Maximum number of times a component of a record was selected: {recs_and_events["count"].iloc[0]} ({recs_and_events["count"].iloc[0] / len(all_recs)*100:.2f} %)")

In [ ]:
recs_and_events

In [ ]:
fig, ax3 = plt.subplots()
ax3.plot(recs_and_events["count"].to_numpy())
ax3.grid(True, which="both", ls="-.", color="0.8")
ax3.minorticks_on()
ax3.set_xlabel("Unique Record")
ax3.set_ylabel("Number of times selected")
ax3.set_ylim(0)
ax3.set_xlim(-10)

fig, ax1 = plt.subplots()
ax1.plot(recs_and_events["cum_%"].to_numpy())
ax1.grid(True, which="both", ls="-.", color="0.8")
ax1.minorticks_on()
ax1.set_xlabel("Unique Record")
ax1.set_ylabel("Cumulative % of all records selected")
ax1.set_ylim(0)
ax1.set_xlim(0)

fig, ax2 = plt.subplots()
ax2.plot(events["cum_%"].to_numpy())
ax2.grid(True, which="both", ls="-.", color="0.8")
ax2.minorticks_on()
ax2.set_xlabel("Unique Event")
ax2.set_ylabel("Cumulative % of all events selected")
ax2.set_ylim(0)
ax2.set_xlim(0)


### Identify the ESM and NGA Records that need to be downloaded

In [ ]:
cols_to_keep = [("metadata", "event_id"), ("metadata", "station_code"), ("metadata", "location_code")]
unique_esm_records = all_recs[all_recs[("metadata", "database")] == "ESM"]\
                     .drop_duplicates(subset=cols_to_keep)
unique_esm_records = unique_esm_records[cols_to_keep].reset_index(drop=True)
esm_records_to_download = unique_esm_records

esm_records_to_download.to_csv(
    cfg["proc_data"]["gm_selection"] / "esm_records_to_download_AvgSA03.csv", index=False)

In [ ]:
# For the NGASub records we need the NGAsubRSN for each combination of event_id and station_code
unique_NGASub_records = all_recs[all_recs[("metadata", "database")] == "NGASub"]\
                     .drop_duplicates(subset=cols_to_keep)
unique_NGASub_ids = list(unique_NGASub_records[[("metadata", "event_id"), ("metadata", "station_code")]].itertuples(index=False, name=None))

# load the NGASub database
ngasub_fp = cfg["raw_data"]["gm_flatfiles"] / "NGASub_Metadata_SA_rotD50.csv"
ngasub_db = pd.read_csv(ngasub_fp, header=0, encoding="cp1252", dtype=str)
new_index = list(ngasub_db[["NGAsubEQID", "NGAsubSSN"]].itertuples(index=False, name=None))
ngasub_db.index = new_index
# ngasub_db = ngasub_db["NGAsubRSN"]

# RSNs to download
ngasub_to_download = ngasub_db.loc[unique_NGASub_ids, ["NGAsubRSN", "Station_Name"]]
ngasub_to_download.to_csv(
    cfg["proc_data"]["gm_selection"] / "ngasub_to_download_AvgSA03.csv", index=False)


# Plotting

In [ ]:
# # Plot the results for the site 31 poe 0.000404
# ensemble = ensembles[(31, 0.0001)]

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))

# trt_format = {
#     "Shallow Default": {"color":"b", "alpha": 1, "lw":1} ,
#     "Subduction Interface": {"color":"0.6", "alpha": 0.5, "lw":1} ,
#     "Subduction Inslab": {"color":"g", "alpha": 1, "lw":1} ,
# }

# periods = [0.01] + [im.period for im in selection_imts if im.name == "SA"]
# sas = ["PGA"] + [im.string for im in selection_imts if im.name == "SA"]
# sa_gcim_qs = ensemble["gcim_quantiles"].loc[sas, :]
# sa_empi_qs = ensemble["quantiles"].loc[sas, :]
# recs = ensemble["recs"]

# # Figure 1 Conditional Spectra
# for ii, (_, row) in enumerate(recs["ims_scaled"].iterrows()):
#     trt = ensemble["ctxs"][ii].trt
#     sa_rec = row[sas]
#     if trt != "Shallow Default":
#         alpha = 0.6
#     else:
#         alpha = 1.0
#     ax1.loglog(periods, sa_rec, **trt_format[trt])

# ax1.loglog(periods, sa_gcim_qs["p16"], color="k", ls="-.")
# ax1.loglog(periods, sa_gcim_qs["p50"], color="k")
# ax1.loglog(periods, sa_gcim_qs["p84"], color="k", ls="-.")

# ax1.loglog(periods, sa_empi_qs["p16"], color="r", ls="-.")
# ax1.loglog(periods, sa_empi_qs["p50"], color="r")
# ax1.loglog(periods, sa_empi_qs["p84"], color="r", ls="-.")

# formatter = ticker.FuncFormatter(custom_log_formatter)
# ax1.xaxis.set_major_formatter(formatter)
# ax1.set_xlim(0.01, 3)
# ax1.set_ylim(1e-2, 5)

# ax1.grid(True, which="both", ls="-.", color="0.8")
# ax1.minorticks_on()
# ax1.tick_params(axis='y', which='minor', left=False)
# ax1.set_xlabel("Period, [s]")
# ax1.set_ylabel("SA [g]")

# # Figure 2 -> RSD595 KS-Test
# im = "RSD595"
# # target cdf and ks bounds
# ks_bounds = ensemble["ks_bounds"][im]
# xs = ks_bounds[:, 0]
# lower = ks_bounds[:, 1]
# cdf = ks_bounds[:, 2]
# upper = ks_bounds[:, 3]

# # ecdf of records
# im_ecdf = ensemble["ecdfs"][im]

# ax2.plot(xs, lower, color="0.8", ls="--")
# ax2.plot(xs, cdf, color="0.8", ls="-")
# ax2.plot(xs, upper, color="0.8", ls="--")
# ax2.plot(im_ecdf[:,0], im_ecdf[:,1], color="b")
# ax2.set_xlim(0, 80)
# ax2.set_ylim(0, 1.0)
# ax2.set_xlabel(r"$D_{s,5\%-95\%}$ [s]")
# ax2.set_ylabel(r"CDF")
# ax1.grid(True, which="both", ls="-.", color="0.8")
# ax1.minorticks_on()
# fig.tight_layout()


In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3), sharex=True,  sharey=True)

# trt_format = {
#     "Shallow Default": {"color":"b", "alpha": 1, "lw":1} ,
#     "Subduction Interface": {"color":"0.6", "alpha": 0.5, "lw":1} ,
#     "Subduction Inslab": {"color":"g", "alpha": 1, "lw":1} ,
# }

# periods = [0.01] + [im.period for im in selection_imts if im.name == "SA"]
# sas = ["PGA"] + [im.string for im in selection_imts if im.name == "SA"]
# # sa_gcim_qs = ensemble["gcim_quantiles"].loc[sas, :]
# # sa_empi_qs = ensemble["quantiles"].loc[sas, :]
# recs = ensemble["recs"]["ims_scaled"]
# sims = ensemble["sims"]

# # Figure 1 Conditional Spectra
# ii = 3
# sa_rec = recs.iloc[ii, :][sas]
# sa_sim = sims.loc[sas, ii]
# ax1.loglog(periods, sa_sim, color="b", label="simulation")
# ax1.loglog(periods, sa_rec, color="r", label="record")
# ax1.legend()

# formatter = ticker.FuncFormatter(custom_log_formatter)
# ax1.xaxis.set_major_formatter(formatter)
# ax1.set_xlim(0.01, 3)
# ax1.set_ylim(1e-2, 5)

# ax1.grid(True, which="both", ls="-.", color="0.8")
# ax1.minorticks_on()
# ax1.tick_params(axis='y', which='minor', left=False)
# ax1.set_xlabel("Period, [s]")
# ax1.set_ylabel("SA [g]")

# # Figure 1 Conditional Spectra
# ii = 0
# sa_rec = recs.iloc[ii, :][sas]
# sa_sim = sims.loc[sas, ii]
# ax2.loglog(periods, sa_sim, color="b")
# ax2.loglog(periods, sa_rec, color="r")

# ax2.grid(True, which="both", ls="-.", color="0.8")
# ax2.minorticks_on()
# ax2.tick_params(axis='y', which='minor', left=False)
# ax2.set_xlabel("Period, [s]")

In [ ]:
# # Plot the results for the site 52 poe 0.0001
# rsr=record_selection_results[("high", 4)][52][0.000201]
# rscores = [e["R-score"] for e in site_rsr["all_ensembles"]]
# ensemble = site_rsr["all_ensembles"][rscores.index(min(rscores))]

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))

# trt_format = {
#     "Shallow Default": {"color":"b", "alpha": 1, "lw":1} ,
#     "Subduction Interface": {"color":"y", "alpha": 0.5, "lw":1} ,
#     "Subduction Inslab": {"color":"g", "alpha": 1, "lw":1} ,
#     "Non-Subduction Deep": {"color":"0.8", "alpha": 1, "lw":1}
# }

# periods = [0.01] + [im.period for im in selection_imts if im.name == "SA"]
# sas = ["PGA"] + [im.string for im in selection_imts if im.name == "SA"]
# # sa_gcim_qs = ensemble["gcim_quantiles"].loc[sas, :]
# sa_empi_qs = ensemble["quantiles"].loc[sas, :]
# recs = ensemble["recs"]

# # Figure 1 Conditional Spectra
# for ii, (_, row) in enumerate(recs["ims_scaled"].iterrows()):
#     trt = ensemble["ctxs"][ii].trt
#     sa_rec = row[sas]
#     if trt != "Shallow Default":
#         alpha = 0.6
#     else:
#         alpha = 1.0
#     ax1.loglog(periods, sa_rec, **trt_format[trt])

# # ax1.loglog(periods, sa_gcim_qs["p16"], color="k", ls="-.")
# # ax1.loglog(periods, sa_gcim_qs["p50"], color="k")
# # ax1.loglog(periods, sa_gcim_qs["p84"], color="k", ls="-.")

# ax1.loglog(periods, sa_empi_qs["p16"], color="r", ls="-.")
# ax1.loglog(periods, sa_empi_qs["p50"], color="r")
# ax1.loglog(periods, sa_empi_qs["p84"], color="r", ls="-.")

# formatter = ticker.FuncFormatter(custom_log_formatter)
# ax1.xaxis.set_major_formatter(formatter)
# ax1.set_xlim(0.01, 3)
# ax1.set_ylim(1e-2, 5)

# ax1.vlines(0.041, 1e-2, 5, color="g", ls=":")

# ax1.grid(True, which="both", ls="-.", color="0.8")
# ax1.minorticks_on()
# ax1.tick_params(axis='y', which='minor', left=False)
# ax1.set_xlabel("Period, [s]")
# ax1.set_ylabel("SA [g]")

# # Figure 2 -> RSD595 KS-Test
# im = "PGA"
# # target cdf and ks bounds
# ks_bounds = ensemble["ks_bounds"][im]
# xs = ks_bounds[:, 0]
# lower = ks_bounds[:, 1]
# cdf = ks_bounds[:, 2]
# upper = ks_bounds[:, 3]

# # ecdf of records
# im_ecdf = ensemble["ecdfs"][im]

# ax2.plot(xs, lower, color="0.8", ls="--")
# ax2.plot(xs, cdf, color="0.8", ls="-")
# ax2.plot(xs, upper, color="0.8", ls="--")
# ax2.plot(im_ecdf[:,0], im_ecdf[:,1], color="b")
# ax2.set_xlim(0, 1.5)
# ax2.set_ylim(0, 1.0)
# ax2.set_xlabel(f"{im}")
# ax2.set_ylabel(r"CDF")
# ax1.grid(True, which="both", ls="-.", color="0.8")
# ax1.minorticks_on()
# fig.tight_layout()


In [ ]:
# ('high', 0), 30, np.float64(0.019801))

In [ ]:
# ensemble_data = records["best_ensemble"]
# ecdfs = ensemble_ecdfs(ensemble_data["recs"]["ims_scaled"])
# ensemble_data["ecdfs"] = ecdfs
# print(ensemble_data.keys())
# print(ecdfs.keys())

In [ ]:
# gcim_data = gcim_AvgSA_03[("high", 0)][31][0.000201]
# gcim_data["stats"]

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker
# import matplotlib.lines as mlines

# from phd_project.plotting import custom_log_formatter

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))

# trt_format = {
#     "Shallow Default": {"color":"b", "alpha": 0.8, "lw":1} ,
#     "Subduction Interface": {"color":"r", "alpha": 0.5, "lw":1} ,
#     "Subduction Inslab": {"color":"g", "alpha": 0.8, "lw":1} ,
# }

# periods = [0.01] + [im.period for im in selection_imts if im.name == "SA"]
# sas = ["PGA"] + [im.string for im in selection_imts if im.name == "SA"]
# sa_dists = gcim_data["stats"].loc[sas, :]
# recs = ensemble_data["recs"]

# # Figure 1 Conditional Spectra
# for ii, (_, row) in enumerate(recs["ims_scaled"].iterrows()):
#     trt = ensemble_data["ctxs"][ii].trt
#     sa_rec = row[sas]
#     if trt != "Shallow Default":
#         alpha = 0.6
#     else:
#         alpha = 1.0
#     ax1.loglog(periods, sa_rec, **trt_format[trt])

# ax1.loglog(periods, sa_dists["p5"], color="k", ls="--")
# ax1.loglog(periods, sa_dists["p16"], color="k", ls="-.")
# ax1.loglog(periods, sa_dists["p50"], color="k")
# ax1.loglog(periods, sa_dists["p84"], color="k", ls="-.")
# ax1.loglog(periods, sa_dists["p95"], color="k", ls="--")

# formatter = ticker.FuncFormatter(custom_log_formatter)
# ax1.xaxis.set_major_formatter(formatter)
# ax1.set_xlim(0.01, 8)
# ax1.set_ylim(1e-2, 5)

# ax1.grid(True, which="both", ls="-.", color="0.8")
# ax1.minorticks_on()
# ax1.tick_params(axis='y', which='minor', left=False)
# ax1.set_xlabel("Period, [s]")
# ax1.set_ylabel("SA [g]")

# # Figure 2 -> RSD595 KS-Test
# im = "RSD595"
# # target cdf and ks bounds
# ks_bounds = records["ks_bounds"][im]
# xs = ks_bounds[:, 0]
# lower = ks_bounds[:, 1]
# cdf = ks_bounds[:, 2]
# upper = ks_bounds[:, 3]

# # ecdf of records
# im_ecdf = ecdfs[im]

# ax2.plot(xs, lower, color="0.8", ls="--")
# ax2.plot(xs, cdf, color="0.8", ls="-")
# ax2.plot(xs, upper, color="0.8", ls="--")
# ax2.plot(im_ecdf[:,0], im_ecdf[:,1], color="b")
# ax2.set_xlim(0, 120)
# ax2.set_ylim(0, 1.0)
# ax2.set_xlabel(r"$D_{s,5\%-95\%}$ [s]")
# ax2.set_ylabel(r"P[IM <= im]")
# ax1.grid(True, which="both", ls="-.", color="0.8")
# ax1.minorticks_on()
# fig.tight_layout()


In [ ]:
# # The trt stats
# trt_stats = pd.DataFrame(recs.groupby(("metadata", "trt")).size())
# trt_stats.columns = ["count"]
# trt_stats["proportions"] = np.round(trt_stats["count"] / trt_stats["count"].sum(), 3)
# trt_stats

In [ ]:
# # Table of KS results
# data = {im : {"ks_statistic": ksr.statistic * ksr.statistic_sign, 
#               "pvalue": ksr.pvalue, 
#               "location": ksr.statistic_location} 
#         for im , ksr in ensemble_data["ks_results"].items()}

# ks_data = pd.DataFrame.from_dict(data, orient="index")
# ks_data

In [ ]:
# # Record data
# metadata = recs["metadata"]
# record_data_for_table = metadata[["event_id", "database", "station_code", "trt", "mag", "rjb", "vs30", "component"]]
# record_data_for_table

In [ ]:
# fp = Path(r"C:\Users\clemettn\Documents\phd\data_processed\06_gm_selection")
# site = 30
# rakes = [0, 90, -90]

# site_records = {}
# site_records_sflim = {}

# for rake in rakes:
#     with open(fp / f"site{site}_record_selections_rake{rake}.pickle", "rb") as file:
#         site_records[rake] = pickle.load(file)

#     with open(fp / f"site{site}_record_selections_rake{rake}_sf_limit.pickle", "rb") as file:
#         site_records_sflim[rake] = pickle.load(file)

In [ ]:
# im = "RSD595"
# poe = 0.0001
# fig, ax = plt.subplots()

# for rake, c in zip(rakes, ["r", "b", "g"]):
#     rs = site_records[rake][poe]
#     ensemble = rs["best_ensemble"]
#     ecdfs = ensemble_ecdfs(ensemble["recs"]["ims_scaled"])

#     # target cdf and ks bounds
#     ks_bounds = rs["ks_bounds"][im]
#     xs = ks_bounds[:, 0]
#     lower = ks_bounds[:, 1]
#     cdf = ks_bounds[:, 2]
#     upper = ks_bounds[:, 3]

#     # ecdf of records
#     im_ecdf = ecdfs1[im]

#     ax.plot(xs, lower, ls="--", color=c, lw=1.0)
#     ax.plot(xs, cdf, ls="-", color=c, lw=1.0)
#     ax.plot(xs, upper, ls="--", color=c, lw=1.0)
#     # ax.plot(im_ecdf[:,0], im_ecdf[:,1], color="b")
#     # ax1.grid(True, which="both", ls="-.", color="0.8")
#     # ax1.minorticks_on()

#     ax.set_xlim(0, 40)
#     ax.set_ylim(0, 1.0)
#     # ax2.set_xlabel(r"$D_{s,5\%-95\%}$ [s]")
#     ax.set_ylabel(r"P[IM <= im]")
#     # ax2.grid(True, which="both", ls="-.", color="0.8")
#     # ax2.minorticks_on()
#     fig.tight_layout()